In [2]:
# 1. Install the missing dependency
!apt-get install -y zstd

# 2. Re-run the Ollama installer
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start the Ollama server in the background
# We use nohup and & to keep it running while we use the notebook
!nohup ollama serve > ollama.log 2>&1 &

# 4. Pull the specific model required by the instructions
!ollama pull llama3.2:1b

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (555 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Addin

In [3]:
import subprocess
import time

# 1. Start Ollama server in the background
# Using subprocess.Popen allows it to persist without blocking the cell
print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 2. Wait for server to wake up
time.sleep(5)

# 3. Pull the model
print("Pulling Llama 3.2-1B...")
!ollama pull llama3.2:1b

# 4. Verify
print("\nVerifying installation:")
!ollama list

Starting Ollama server...
Pulling Llama 3.2-1B...


Verifying installation:
NAME           ID              SIZE      MODIFIED               
llama3.2:1b    baf6a787fdff    1.3 GB    Less than a second ago    


In [4]:
%%writefile program1.py
import requests
import json
from datasets import load_dataset
from tqdm.auto import tqdm
from datetime import datetime

# ============================================================================
# CONFIGURATION
# ============================================================================
MODEL_NAME = "llama3.2:1b"
MMLU_SUBJECTS = ["abstract_algebra"] # Set a unique subject for each program

def format_mmlu_prompt(question, choices):
    choice_labels = ["A", "B", "C", "D"]
    prompt = f"{question}\n\n"
    for label, choice in zip(choice_labels, choices):
        prompt += f"{label}. {choice}\n"
    prompt += "\nAnswer:"
    return prompt

def get_model_prediction(prompt):
    url = "http://localhost:11434/api/generate"
    data = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {"num_predict": 1, "temperature": 0}
    }
    try:
        response = requests.post(url, json=data)
        generated_text = response.json().get("response", "")
        return generated_text.strip()[:1].upper()
    except Exception as e:
        return "A" # Fallback

def main():
    print(f"Starting evaluation on: {MMLU_SUBJECTS[0]}")
    results = []

    for subject in MMLU_SUBJECTS:
        dataset = load_dataset("cais/mmlu", subject, split="test")
        correct, total = 0, 0

        for example in tqdm(dataset, desc=f"Testing {subject}"):
            prompt = format_mmlu_prompt(example["question"], example["choices"])
            prediction = get_model_prediction(prompt)
            if prediction == ["A", "B", "C", "D"][example["answer"]]:
                correct += 1
            total += 1

        print(f"Result for {subject}: {correct}/{total}")

if __name__ == "__main__":
    main()

Writing program1.py


In [5]:
%%writefile program2.py
import requests
import json
from datasets import load_dataset
from tqdm.auto import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================
MODEL_NAME = "llama3.2:1b"
# Program 2 uses a different topic: College Computer Science
MMLU_SUBJECTS = ["college_computer_science"]

def format_mmlu_prompt(question, choices):
    choice_labels = ["A", "B", "C", "D"]
    prompt = f"{question}\n\n"
    for label, choice in zip(choice_labels, choices):
        prompt += f"{label}. {choice}\n"
    prompt += "\nAnswer:"
    return prompt

def get_model_prediction(prompt):
    url = "http://localhost:11434/api/generate"
    data = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": 1,
            "temperature": 0
        }
    }
    try:
        response = requests.post(url, json=data)
        generated_text = response.json().get("response", "")
        # Clean the output to get just the letter
        answer = generated_text.strip()[:1].upper()
        return answer if answer in ["A", "B", "C", "D"] else "A"
    except Exception:
        return "A"

def main():
    print(f"Starting evaluation on: {MMLU_SUBJECTS[0]}")

    for subject in MMLU_SUBJECTS:
        try:
            dataset = load_dataset("cais/mmlu", subject, split="test")
            correct, total = 0, 0

            for example in tqdm(dataset, desc=f"Testing {subject}"):
                prompt = format_mmlu_prompt(example["question"], example["choices"])
                prediction = get_model_prediction(prompt)

                correct_answer = ["A", "B", "C", "D"][example["answer"]]
                if prediction == correct_answer:
                    correct += 1
                total += 1

            accuracy = (correct / total * 100) if total > 0 else 0
            print(f"\nFinal Result for {subject}: {correct}/{total} ({accuracy:.2f}%)")
        except Exception as e:
            print(f"Error evaluating {subject}: {e}")

if __name__ == "__main__":
    main()

Writing program2.py


In [6]:
%%bash
time { python program1.py ; python program2.py ; }

Starting evaluation on: abstract_algebra
Result for abstract_algebra: 0/100
Starting evaluation on: college_computer_science

Final Result for college_computer_science: 26/100 (26.00%)


Testing college_computer_science: 100%|██████████| 100/100 [00:32<00:00,  3.03it/s]

real	1m16.016s
user	0m4.351s
sys	0m0.436s


In [7]:
%%bash
time { python program1.py & python program2.py & wait ; }

Starting evaluation on: abstract_algebra
Starting evaluation on: college_computer_science
Result for abstract_algebra: 0/100

Final Result for college_computer_science: 26/100 (26.00%)


Testing college_computer_science: 100%|██████████| 100/100 [01:01<00:00,  1.64it/s]

real	1m4.422s
user	0m4.529s
sys	0m0.476s
